In [1]:
from snowflake.snowpark import Session
from snowflake.ml.jobs import submit_directory
from poc.utils import create_session, get_connection_config, get_training_config, get_inference_config

conn_cfg = get_connection_config()
train_cfg = get_training_config()
infer_cfg = get_inference_config()

session = create_session()

In [2]:
from datetime import datetime, timezone
train_start = datetime.now(timezone.utc)
train_job = submit_directory(
    dir_path="poc/",
    compute_pool=train_cfg["compute_pool_name"],
    entrypoint="train.py",
    stage_name=conn_cfg["stage_artifacts"],
    target_instances=train_cfg["target_cluster_size"],
    session=session,
)
train_job.wait()
train_end = datetime.now(timezone.utc)
print(train_job.get_logs())

Compute pool busy (1/5 nodes in use, 5 nodes required). Job execution may be delayed.


Connected: "UR20380"
🔄 Preparing data for 100 partitions
   Test percentage: 10%
   Train rows: 65,700
   Test rows: 7,300
----------------------------------------------------------------------------------------------------------------------------------
|"name"                                              |"size"  |"md5"                             |"last_modified"                |
----------------------------------------------------------------------------------------------------------------------------------
|ml_stage/train_features/P0000/data_01c31804-000...  |30753   |76e8e84549cb2ed33ce826ae74487975  |Tue, 17 Mar 2026 18:44:05 GMT  |
|ml_stage/train_features/P0001/data_01c31804-000...  |31425   |9c0bfaf6e5532a90e86c2608f12b417d  |Tue, 17 Mar 2026 18:44:05 GMT  |
|ml_stage/train_features/P0002/data_01c31804-000...  |30790   |30b8d91bd28f11c8727c5cf4e28fddf1  |Tue, 17 Mar 2026 18:44:05 GMT  |
|ml_stage/train_features/P0003/data_01c31804-000...  |31377   |0af9dae31be9c106c2143270ff0a

In [3]:
infer_start = datetime.now(timezone.utc)
infer_job = submit_directory(
    dir_path="poc/",
    compute_pool=infer_cfg["compute_pool_name"],
    entrypoint="infer.py",
    stage_name=conn_cfg["stage_artifacts"],
    target_instances=infer_cfg["target_cluster_size"],
    session=session,
)
infer_job.wait()
infer_end = datetime.now(timezone.utc)
print(infer_job.get_logs())

Compute pool busy (1/5 nodes in use, 5 nodes required). Job execution may be delayed.


Connected: "UR20380"
📋 Using models from: training_v20260317_1843
   Inference run ID:  inference_20260317_1848
----------------------------------------------------------------------------------------------------------------------------------
|"name"                                              |"size"  |"md5"                             |"last_modified"                |
----------------------------------------------------------------------------------------------------------------------------------
|ml_stage/infer_data/P0000/data_01c31808-0004-c9...  |6491    |23575014273cd7def1ee55c753254459  |Tue, 17 Mar 2026 18:48:30 GMT  |
|ml_stage/infer_data/P0001/data_01c31808-0004-c9...  |6485    |b7068d64498482e7c58ae1884e8465fd  |Tue, 17 Mar 2026 18:48:30 GMT  |
|ml_stage/infer_data/P0002/data_01c31808-0004-c9...  |6487    |19deb2d96db8f8ce93cb430b0c87cd8e  |Tue, 17 Mar 2026 18:48:30 GMT  |
|ml_stage/infer_data/P0003/data_01c31808-0004-c9...  |6498    |fde74b57a7707576c9497ac00dbc4eaa  |Tue,

## Job Metrics: Time and Credit Consumption ESTIMATION

In [4]:
INSTANCE_FAMILY_CREDITS_PER_HOUR = {
    "CPU_X64_XS": 0.06,
    "CPU_X64_S": 0.19,
    "CPU_X64_M": 0.38,
    "CPU_X64_SL": 0.86,
    "CPU_X64_L": 1.72,
    "HIGHMEM_X64_S": 0.38,
    "HIGHMEM_X64_M": 1.72,
    "HIGHMEM_X64_SL": 5.66,
    "HIGHMEM_X64_L": 7.62,
    "GPU_NV_XS": 1.0,
    "GPU_NV_S": 1.5,
    "GPU_NV_SM": 3.0,
    "GPU_NV_M": 6.0,
    "GPU_NV_2M": 9.0,
    "GPU_NV_3M": 16.0,
    "GPU_NV_L": 18.5,
    "GPU_NV_SL": 32.0,
}

WAREHOUSE_SIZE_CREDITS_PER_HOUR = {
    "X-Small": 1, "Small": 2, "Medium": 4, "Large": 8,
    "X-Large": 16, "2X-Large": 32, "3X-Large": 64, "4X-Large": 128,
    "5X-Large": 256, "6X-Large": 512,
}

In [5]:
train_duration = (train_end - train_start).total_seconds()
infer_duration = (infer_end - infer_start).total_seconds()

def calc_container_credits(duration_secs, instance_family, num_nodes):
    rate = INSTANCE_FAMILY_CREDITS_PER_HOUR.get(instance_family, 0)
    return (duration_secs / 3600) * rate * num_nodes

train_container_credits = calc_container_credits(train_duration, train_cfg["instance_family"], train_cfg["target_cluster_size"])
infer_container_credits = calc_container_credits(infer_duration, infer_cfg["instance_family"], infer_cfg["target_cluster_size"])

query_tag = conn_cfg.get("tag", "")

wh_train_query = f"""
SELECT 
    WAREHOUSE_SIZE,
    SUM(EXECUTION_TIME) / 1000 AS total_exec_secs
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE USER_NAME = '{train_job.name}'
  AND WAREHOUSE_SIZE IS NOT NULL
GROUP BY WAREHOUSE_SIZE
"""
wh_train_df = session.sql(wh_train_query).to_pandas()

wh_infer_query = f"""
SELECT 
    WAREHOUSE_SIZE,
    SUM(EXECUTION_TIME) / 1000 AS total_exec_secs
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE USER_NAME = '{infer_job.name}'
  AND WAREHOUSE_SIZE IS NOT NULL
GROUP BY WAREHOUSE_SIZE
"""
wh_infer_df = session.sql(wh_infer_query).to_pandas()

def calc_wh_credits_from_df(df):
    credits = 0
    for _, row in df.iterrows():
        rate = WAREHOUSE_SIZE_CREDITS_PER_HOUR.get(row['WAREHOUSE_SIZE'], 0)
        credits += (row['TOTAL_EXEC_SECS'] / 3600) * rate
    return credits

train_wh_credits = calc_wh_credits_from_df(wh_train_df)
infer_wh_credits = calc_wh_credits_from_df(wh_infer_df)

In [6]:
import pandas as pd

summary_df = pd.DataFrame({
    'Job': ['Training', 'Inference', 'Total'],
    'Instance Family': [train_cfg["instance_family"], infer_cfg["instance_family"], '-'],
    'Nodes': [train_cfg["target_cluster_size"], infer_cfg["target_cluster_size"], '-'],
    'Duration (seconds)': [round(train_duration, 2), round(infer_duration, 2), round(train_duration + infer_duration, 2)],
    'Duration (minutes)': [round(train_duration / 60, 2), round(infer_duration / 60, 2), round((train_duration + infer_duration) / 60, 2)],
    'Container Credits (est)': [round(train_container_credits, 6), round(infer_container_credits, 6), round(train_container_credits + infer_container_credits, 6)],
    'Warehouse Credits (est)': [round(train_wh_credits, 6), round(infer_wh_credits, 6), round(train_wh_credits + infer_wh_credits, 6)],
    'Total Credits (est)': [round(train_container_credits + train_wh_credits, 6), round(infer_container_credits + infer_wh_credits, 6), round(train_container_credits + infer_container_credits + train_wh_credits + infer_wh_credits, 6)]
})
summary_df

,Job,Instance Family,Nodes,Duration (seconds),Duration (minutes),Container Credits (est),Warehouse Credits (est),Total Credits (est)
0,Training,CPU_X64_S,5,272.68,4.54,0.071958,0.008344,0.080303
1,Inference,CPU_X64_S,5,241.96,4.03,0.063849,0.000000,0.063849
2,Total,-,-,514.64,8.58,0.135808,0.008344,0.144152
